In [5]:
"""
Transformer-encoder training script for CriDe.
Shares dataset/split/metrics machinery with train_bilstm.py via data_utils.py --
keep all three files in the same folder.

Needs: torch, numpy, scikit-learn, tqdm
"""
# This file is generated by claude code for training the transformer model 

import os
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from data_utils import (
    set_seed, UCFCrimeDataset, stratified_split, class_weights_from_labels,
    collate_pad, EarlyStopping, compute_metrics, save_checkpoint, save_label_map,
)

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------
DATA_PATH       = "./extracted_features"
SAVE_DIR        = "Transformer2"
INPUT_DIM       = 512
D_MODEL         = 128
NHEAD           = 8
NUM_LAYERS      = 2
DIM_FEEDFORWARD = 256
DROPOUT         = 0.5
BATCH_SIZE      = 64
MAX_EPOCHS      = 150
PEAK_LR         = 1e-4
WARMUP_STEPS    = 500
WEIGHT_DECAY    = 5e-4
VAL_FRACTION    = 0.15
PATIENCE        = 25
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.0
USE_AUX_HEAD    = True
AUX_LOSS_WEIGHT = 0.1
MAX_SEQ_LEN     = 32
NUM_WORKERS     = 2
SEED            = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
os.makedirs(SAVE_DIR, exist_ok=True)
set_seed(SEED)


# ----------------------------------------------------------------
# POSITIONAL ENCODING
# ----------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ----------------------------------------------------------------
# MODEL
# ----------------------------------------------------------------
class CrimeTransformer(nn.Module):
    def __init__(self, input_dim=512, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=512, dropout=0.2, num_classes=14,
                 max_seq_len=256, use_aux_head=True):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_seq_len + 1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.GELU(),
            nn.Dropout(dropout),
        
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
        
            nn.Linear(128, num_classes)
        )
        self.use_aux_head = use_aux_head
        if use_aux_head:
            self.aux_head = nn.Linear(d_model, 1)

    def forward(self, x, pad_mask):
        B = x.size(0)
        x = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
        full_mask = torch.cat([cls_mask, pad_mask], dim=1)
        x = self.pos_enc(x)
        encoded = self.encoder(x, src_key_padding_mask=full_mask)
        features = encoded[:, 1:]  # ignore CLS token

        valid_mask = (~pad_mask).unsqueeze(-1)
        
        pooled = (features * valid_mask).sum(dim=1)
        pooled = pooled / valid_mask.sum(dim=1).clamp(min=1)
        
        pooled = self.norm(pooled)
        logits = self.classifier(pooled)
        aux_logits = self.aux_head(pooled).squeeze(-1) if self.use_aux_head else None
        return logits, aux_logits


# ----------------------------------------------------------------
# DATA
# ----------------------------------------------------------------
full_dataset = UCFCrimeDataset(DATA_PATH)
train_ds, val_ds = stratified_split(full_dataset, VAL_FRACTION, SEED)
num_classes = len(full_dataset.label_map)
normal_idx = full_dataset.find_normal_index()
if USE_AUX_HEAD and normal_idx is None:
    print("Couldn't find a 'Normal' class in your folder names -- disabling the aux head.")
    USE_AUX_HEAD = False
    
# Improvement from chatgpt  
from torch.utils.data import WeightedRandomSampler

train_labels = train_ds.labels

class_counts = torch.bincount(torch.tensor(train_labels))
class_weights_sampler = 1.0 / class_counts.float()

sample_weights = [class_weights_sampler[label].item() for label in train_labels]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_pad
)



# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
#                            num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_pad)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_pad)

class_weights = class_weights_from_labels(train_ds.labels, num_classes, device)

# ----------------------------------------------------------------
# MODEL / OPTIMIZER  (warmup + cosine decay)
# ----------------------------------------------------------------
model = CrimeTransformer(INPUT_DIM, D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD,
                          DROPOUT, num_classes, MAX_SEQ_LEN, USE_AUX_HEAD).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
aux_criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=WEIGHT_DECAY)

total_steps = MAX_EPOCHS * max(1, len(train_loader))

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
stopper = EarlyStopping(patience=PATIENCE, mode="max")
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def make_anomaly_targets(y):
    return (y != normal_idx).float()


def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, all_preds, all_labels = 0.0, [], []
    for x, lengths, pad_mask, y in tqdm(loader, desc="train" if train else "val", leave=False):
        x = x.to(device, non_blocking=True)

        if train:
            x = x + 0.01 * torch.randn_like(x)
        pad_mask = pad_mask.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits, aux_logits = model(x, pad_mask)
                loss = criterion(logits, y)
                if USE_AUX_HEAD:
                    loss = loss + AUX_LOSS_WEIGHT * aux_criterion(aux_logits, make_anomaly_targets(y))
        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        total_loss += loss.item() * x.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    metrics = compute_metrics(all_labels, all_preds, full_dataset.label_map)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


# ----------------------------------------------------------------
# TRAIN LOOP
# ----------------------------------------------------------------
best_path = os.path.join(SAVE_DIR, "transformer_best.pth")
best_report = None
for epoch in range(1, MAX_EPOCHS + 1):
    train_m = run_epoch(train_loader, train=True)
    val_m = run_epoch(val_loader, train=False)
    improved = stopper.step(val_m["macro_f1"])
    print(f"Epoch {epoch:3d} | train loss {train_m['loss']:.4f} acc {train_m['accuracy']:.3f} "
          f"| val loss {val_m['loss']:.4f} acc {val_m['accuracy']:.3f} macroF1 {val_m['macro_f1']:.3f}")
    if improved:
        best_report = val_m["report"]
        save_checkpoint(best_path, model, optimizer, epoch, val_m["macro_f1"],
                         full_dataset.label_map,
                         dict(input_dim=INPUT_DIM, d_model=D_MODEL, nhead=NHEAD,
                              num_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD,
                              num_classes=num_classes, max_seq_len=MAX_SEQ_LEN,
                              use_aux_head=USE_AUX_HEAD))
        print(f"  -> new best (macroF1={val_m['macro_f1']:.3f}), saved to {best_path}")
    # if stopper.should_stop:
    #     print(f"No improvement for {PATIENCE} epochs, stopping early.")
    #     break

save_label_map(os.path.join(SAVE_DIR, "label_map.json"), full_dataset.label_map)
print("\nBest-epoch validation report:\n", best_report)

C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:190: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


Using device: cuda
Classes: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}
Total samples: 1610


train:   0%|          | 0/22 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
val:   0%|          | 0/4 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                                                  

Epoch   1 | train loss 2.6894 acc 0.093 | val loss 2.7176 acc 0.025 macroF1 0.004
  -> new best (macroF1=0.004), saved to Transformer2\transformer_best.pth


train:   0%|          | 0/22 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
val:   0%|          | 0/4 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                                                  

...

Epoch 148 | train loss 0.2607 acc 0.792 | val loss 4.4074 acc 0.145 macroF1 0.157


train:   0%|          | 0/22 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
val:   0%|          | 0/4 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                                                  

Epoch 149 | train loss 0.2481 acc 0.811 | val loss 4.4071 acc 0.145 macroF1 0.158


train:   0%|          | 0/22 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
val:   0%|          | 0/4 [00:00<?, ?it/s]C:\Users\navee\AppData\Local\Temp\ipykernel_38020\3212109122.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                                                  

Epoch 150 | train loss 0.2664 acc 0.795 | val loss 4.4073 acc 0.145 macroF1 0.158

Best-epoch validation report:
                precision    recall  f1-score   support

        Abuse       0.04      0.14      0.06         7
       Arrest       0.09      0.29      0.14         7
        Arson       0.19      0.50      0.27         6
      Assault       0.20      0.29      0.24         7
     Burglary       0.12      0.31      0.17        13
    Explosion       0.00      0.00      0.00         5
     Fighting       0.21      0.43      0.29         7
 NormalVideos       0.00      0.00      0.00       120
RoadAccidents       0.21      0.16      0.18        19
      Robbery       0.33      0.27      0.30        22
     Shooting       0.09      0.25      0.13         4
  Shoplifting       0.04      0.25      0.07         4
     Stealing       0.47      0.50      0.48        14
    Vandalism       0.00      0.00      0.00         7

     accuracy                           0.14       242
    